In [ ]:

%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-chroma
%pip install -U langchain-ollama

In [ ]:
%pip install -U pypdf

In [ ]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

In [ ]:
llm = ChatOllama(
    model="phi3",
    temperature=0
)

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [ ]:
def load_documents(folder_path="docs"):

    documents = []

    folder = Path(folder_path)

    for file in folder.iterdir():

        if file.suffix.lower() == ".pdf":

            loader = PyPDFLoader(str(file))
            documents.extend(loader.load())

        elif file.suffix.lower() == ".txt":

            loader = TextLoader(str(file), encoding="utf-8")
            documents.extend(loader.load())

    print(f"Loaded {len(documents)} document pages")

    return documents

In [ ]:
documents = load_documents()

documents[:2]

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")



In [ ]:
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="db"
)

print("Vector database created")

In [ ]:
retriever = db.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
query = "What is blue?"

In [ ]:
docs = retriever.invoke(query)

len(docs)

In [ ]:
context = "\n\n".join(doc.page_content for doc in docs)

In [ ]:
prompt = f"""
You are a helpful assistant.

Answer ONLY using the context below.

If the answer is not present, say:

"I couldn't find that information in the documents."

Context:

{context}

Question:

{query}

Answer:
"""

In [ ]:
response = llm.invoke(prompt)

print(response.content)